In [1]:
import pandas as pd
import numpy as np

In [2]:
import duckdb

con = duckdb.connect()

In [3]:
con.execute("CREATE OR REPLACE TABLE loans AS SELECT * FROM read_csv_auto('credit_risk_dataset.csv')")

con.execute("DESCRIBE loans").df()

,column_name,column_type,null,key,default,extra
0,person_age,BIGINT,YES,None,None,None
1,person_income,BIGINT,YES,None,None,None
2,person_home_ownership,VARCHAR,YES,None,None,None
3,person_emp_length,DOUBLE,YES,None,None,None
4,loan_intent,VARCHAR,YES,None,None,None
5,loan_grade,VARCHAR,YES,None,None,None
6,loan_amnt,BIGINT,YES,None,None,None
7,loan_int_rate,DOUBLE,YES,None,None,None
8,loan_status,BIGINT,YES,None,None,None
9,loan_percent_income,DOUBLE,YES,None,None,None


In [4]:
import getpass
from google import genai
from google.genai import types

client = genai.Client(api_key=getpass.getpass("Gemini API key: "))

In [5]:
schema = """
Table: loans (credit risk data, one row per loan applicant)
Columns:
- person_age (int, years)
- person_income (int, annual income)
- person_home_ownership (text: RENT, OWN, MORTGAGE, OTHER)
- person_emp_length (float, years employed, can be NULL)
- loan_intent (text: EDUCATION, MEDICAL, VENTURE, PERSONAL, DEBTCONSOLIDATION, HOMEIMPROVEMENT)
- loan_grade (text: A to G, A is the lowest risk)
- loan_amnt (int, loan amount)
- loan_int_rate (float, interest rate in percent, can be NULL)
- loan_status (int: 1 = defaulted, 0 = did not default)
- loan_percent_income (float, loan amount as a fraction of income)
- cb_person_default_on_file (text: Y or N, previous default on credit bureau record)
- cb_person_cred_hist_length (int, credit history length in years)
"""

def question_to_sql(question):
    response = client.models.generate_content(
        model="gemini-3-flash-preview",
        contents=question,
        config=types.GenerateContentConfig(
            system_instruction=f"You write DuckDB SQL. Use only this schema:\n{schema}\nReturn only the SQL query, with no explanation and no markdown."
        )
    )
    return response.text.strip()

In [6]:
def is_safe(sql):
    s = sql.strip().rstrip(";").lower()
    banned = ["insert", "update", "delete", "drop", "alter", "create", "attach", "copy", "pragma"]
    return s.startswith("select") and ";" not in s and not any(w in s for w in banned)

def clean_sql(sql):
    return sql.replace("```sql", "").replace("```", "").strip()

def ask(question):
    sql = clean_sql(question_to_sql(question))
    if not is_safe(sql):
        return sql, None, "Blocked: only read-only SELECT queries are allowed"

    try:
        df = con.execute(sql).df()
    except Exception as e:
        return sql, None, f"The generated SQL failed: {e}"

    summary = client.models.generate_content(
        model = "gemini-3-flash-preview",
        contents = f"Question: {question}\nSQL result:\n{df.head(20).to_string()}\nAnswer in 2-3 plain-English sentences."
    ).text

    return sql, df, summary

In [7]:
sql, df, summary = ask("What is the default rate by loan grade?")
print(sql)
display(df)
print(summary)

SELECT loan_grade, AVG(loan_status) AS default_rate FROM loans GROUP BY loan_grade ORDER BY loan_grade;


,loan_grade,default_rate
0,A,0.099564
1,B,0.162760
2,C,0.207340
3,D,0.590458
4,E,0.644191
5,F,0.705394
6,G,0.984375


The default rate increases significantly as the loan grade moves from A to G, ranging from approximately 10% for Grade A to over 98% for Grade G. There is a sharp rise in risk starting at Grade D, where the default rate jumps to 59% compared to the lower rates seen in Grades A through C.
